# Signal exploration: what the data says before modeling

This notebook is the exploration that shaped the Phase 1 decisions. It asks one question from several angles: **do popularity signals (Billboard charts, Wikipedia pageviews, Spotify) line up with how well a show sells, and how?**

Every result here reads from the SQLite database built by the pipeline, so it can be rerun from scratch:

```
python -m concert_demand.load_db
python -m concert_demand.features.signals
python -m concert_demand.features.one_hit
python -m concert_demand.analysis_views
python -m concert_demand.clean.production
```

All signals are measured **six months before each show**, using only data from before that date, so nothing here leaks information a talent buyer wouldn't have had when booking.

In [ ]:
import sqlite3

import numpy as np
import pandas as pd
from scipy.stats import spearmanr
from sklearn.metrics import roc_auc_score

from concert_demand.load_db import DB_PATH

pd.set_option("display.width", 160)
connection = sqlite3.connect(DB_PATH)

# The modeling sample: US + Canada shows with clean data and a full 52-week signal window
shows = pd.read_sql(
    """SELECT f.*, m.sold_out, m.fill_rate, m.capacity_per_show, m.covid_era, m.tour,
              1.0 * m.tickets_sold / m.n_shows AS tickets_per_show
       FROM show_features AS f
       JOIN model_ready_engagements AS m USING (engagement_id)
       WHERE f.in_model_window = 1""",
    connection,
)
print(f"{len(shows):,} shows, {shows['artist'].nunique()} artists, {shows['tour'].nunique()} tours")
print(f"Sellout rate: {shows['sold_out'].mean():.1%}")

## 1. Which signals carry any information?

Each feature is scored three ways:

- **AUC for sellouts.** The chance that a randomly picked sold-out show has a higher value than a randomly picked show that didn't sell out. 0.5 is a coin flip, 1.0 is perfect.
- **Fill rate when not sold out.** Rank correlation with fill rate among shows that didn't sell out (the second part of the two-part model).
- **Tickets per show.** Rank correlation with raw demand.

Best rank is flipped so that a better chart position counts as a higher value.

In [ ]:
signal_columns = [c for c in shows.columns
                  if c.startswith(("hot100", "billboard200", "pageviews", "spotify"))]
rows = []
for column in signal_columns:
    data = shows[[column, "sold_out", "fill_rate", "tickets_per_show"]].dropna()
    if data[column].nunique() < 2:
        continue  # e.g. Billboard 200 featured credits: albums almost never credit features
    value = -data[column] if "best_rank" in column else data[column]
    unsold = data["sold_out"] == 0
    rows.append({
        "signal": column,
        "shows": len(data),
        "auc_sellout": roc_auc_score(data["sold_out"], value),
        "rho_fill_unsold": spearmanr(value[unsold], data.loc[unsold, "fill_rate"])[0],
        "rho_tickets": spearmanr(value, data["tickets_per_show"])[0],
    })
ranking = pd.DataFrame(rows).sort_values("auc_sellout", ascending=False)
ranking.head(15).round(3)

**Reading it:** every signal is weak on its own (the best AUCs sit around 0.6). Career chart depth beats recent chart runs, and pageview momentum adds almost nothing. No single signal predicts sellouts well, which is why the model will combine them.

## 2. Is the pageview signal just a few superstars?

The top pageview decile sells out far more often than the rest. Two ways that could be misleading: a handful of superstars driving it, or the post-COVID boom.

In [ ]:
shows["pageview_decile"] = pd.qcut(shows["pageviews_mean_52w"], 10, labels=False) + 1
by_decile = shows.groupby("pageview_decile").agg(
    shows=("sold_out", "size"),
    artists=("artist", "nunique"),
    sellout_rate=("sold_out", "mean"),
)
top = shows[shows["pageview_decile"] == 10]
biggest = top["artist"].value_counts().index[0]
print(f"Top decile without its biggest artist ({biggest}): "
      f"{top.loc[top['artist'] != biggest, 'sold_out'].mean():.1%} sellouts")
by_decile.round(3)

In [ ]:
# Same table split by COVID era: does the top decile stand out in both?
pd.crosstab(shows["pageview_decile"], shows["covid_era"],
            values=shows["sold_out"], aggfunc="mean").round(3)

**Reading it:** the effect survives both checks. It isn't one artist and it isn't the post-COVID boom.

## 3. Do pageviews say *who* is popular, or *when*?

If an artist's attention rising above their own normal level predicted sellouts, the signal would help with timing. The `within_artist_pageviews` view compares each show's pageviews to that artist's own average.

In [ ]:
pd.read_sql("SELECT * FROM within_artist_pageviews", connection).round(3)

**Reading it:** no clear pattern from the lowest quintile to the highest. Pageviews separate big artists from small ones, but don't say when a given artist is hot.

## 4. Do talent buyers already size rooms to popularity?

If buyers book popular artists into bigger rooms, popularity should line up with **tickets per show** more strongly than with **sellouts**. Ranking table (section 1) and the demand-per-seat view below test this.

In [ ]:
print(ranking.set_index("signal").loc[
    ["pageviews_mean_52w", "hot100_lead_entries_career", "billboard200_lead_points_52w"],
    ["auc_sellout", "rho_tickets"]].round(3))
pd.read_sql("SELECT * FROM demand_per_seat", connection).round(3)

**Reading it:** the signals track tickets per show better than sellouts, and popularity per seat barely moves the sellout rate. That fits buyers already pricing popularity into the room size, and it's why Phase 2 compares the sellout model with a demand model (tickets per show against capacity).

## 5. Sellouts cluster by tour

If one show on a tour sells out, the others usually do too. That matters for model evaluation: shows from the same tour in both training and test data would make accuracy look better than it is.

In [ ]:
tour_rates = shows.groupby("tour")["sold_out"].mean()
print(f"{len(tour_rates)} tours")
print(f"Every show sold out:  {(tour_rates == 1).mean():.1%} of tours")
print(f"No show sold out:     {(tour_rates == 0).mean():.1%} of tours")

**Reading it:** about a third of tours sold out every date. Phase 2 will use cross-validation grouped by artist, so the model is always tested on artists it hasn't seen.

## 6. The Spanish Wikipedia fix

English Wikipedia undercounted Latin artists, whose fans often read the Spanish article. Comparing the English-only and combined signals for Latin artists:

In [ ]:
pd.read_sql(
    """SELECT f.signal_artist AS artist,
              ROUND(AVG(f.pageviews_en_mean_52w)) AS english_only,
              ROUND(AVG(f.pageviews_mean_52w))    AS english_plus_spanish
       FROM show_features AS f JOIN artists AS a ON a.artist = f.signal_artist
       WHERE a.genre_group = 'Latin' AND f.in_model_window = 1
       GROUP BY f.signal_artist
       ORDER BY english_plus_spanish DESC
       LIMIT 8""", connection)

## 7. One-hit acts: booked bigger than their fan base?

Matt's definition: one song supplies at least 70% of recent chart points, the artist drew under 3,000 daily pageviews before that song, and has little album success over their career.

In [ ]:
case_study = pd.read_sql("SELECT * FROM one_hit_case_study", connection)
acts = pd.read_sql(
    """SELECT f.signal_artist AS artist, o.top_song, COUNT(*) AS shows,
              AVG(m.sold_out) AS sellout_rate, AVG(m.fill_rate) AS fill_rate
       FROM one_hit_shows AS o
       JOIN show_features AS f USING (engagement_id)
       JOIN model_ready_engagements AS m USING (engagement_id)
       WHERE o.is_one_hit = 1
       GROUP BY f.signal_artist, o.top_song""", connection)
display(case_study.round(3))
acts.round(3)

**Reading it:** only two acts qualify, with opposite outcomes. Kim Petras sold 42% of her seats and never sold out; Tate McRae sold out every date. Too few for a model feature, but a clear case study: two acts that looked alike at booking.

## 8. Production level: a first, unadjusted look

Production level comes from Matt's rubric (Standard, Enhanced, Themed), labeled from each tour's Wikipedia article. Most articles don't describe the show, and those that do skew toward notable productions.

In [ ]:
pd.read_sql(
    """SELECT p.production_level, COUNT(*) AS shows, COUNT(DISTINCT m.tour) AS tours,
              ROUND(AVG(m.sold_out), 3) AS sellout_rate
       FROM model_sample AS m JOIN tour_production AS p USING (tour)
       GROUP BY p.production_level
       ORDER BY sellout_rate DESC""", connection)

**Reading it:** enhanced and themed shows sell out more often, but bigger productions belong to bigger artists, so this isn't adjusted for popularity or genre yet. Phase 2 tests it with those controls, and compares production level against what's normal for the artist's genre.

## Takeaways for Phase 2

1. No single signal is strong. The model needs to combine them, and career depth matters more than recent hits.
2. Pageviews identify *who* is popular, not *when*. Timing signals need to come from elsewhere.
3. Buyers already size rooms to popularity, so predicting demand (tickets per show) and comparing it to capacity may be the more useful framing for right-sizing.
4. Sellouts cluster by tour, so evaluation must be grouped by artist.
5. One-hit acts and production level are case studies or controlled tests, not headline features.